# 20ftコンテナ向け3次元バンニングレイアウト

ケースリストを読み込み、制約を満たす配置をヒューリスティックで生成し、評価用JSONを出力します。

In [1]:
# セル1: ライブラリ導入
from __future__ import annotations

import json
import math
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

try:
    import openpyxl  # noqa: F401
except ImportError:
    openpyxl = None

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


In [2]:
# セル2: 定数定義
CONTAINER_LENGTH_MM = 5898
CONTAINER_WIDTH_MM = 2352
CONTAINER_HEIGHT_MM = 2393
MAX_CONTAINER_WEIGHT_KG = 12000.0
COG_RADIUS_LIMIT_MM = 300.0
ROTATIONS = [0, 90]

CONTAINER_CENTER_X_MM = CONTAINER_LENGTH_MM / 2.0
CONTAINER_CENTER_Y_MM = CONTAINER_WIDTH_MM / 2.0
CONTAINER_VOLUME_MM3 = CONTAINER_LENGTH_MM * CONTAINER_WIDTH_MM * CONTAINER_HEIGHT_MM

# Colab想定の既定パス。必要なら手で変更する。
INPUT_PATH: Optional[Path] = Path('/content/drive/MyDrive/バンニングレイアウト/(抜粋)ケースリスト.csv')
OUTPUT_JSON_PATH = Path('output_solution.json')
INSTANCE_NAME: Optional[str] = None
ALGORITHM_NAME = 'ffd_3d_single_support_centroid_v2'


In [3]:
# セル3: Excel読込
def normalize_column_name(name: object) -> str:
    text = str(name).strip().lower()
    text = text.replace('　', '').replace(' ', '')
    text = text.replace('_', '').replace('-', '')
    text = text.replace('（', '(').replace('）', ')')
    return text


COLUMN_CANDIDATES = {
    'case_id': ['caseid', 'case_no', 'casecode', 'id', 'ケースid', 'ケース番号', 'ケースno', 'no', '№', '番号'],
    'length': ['length', 'len', 'l', '全長', '長さ', '寸法l'],
    'width': ['width', 'wid', 'w', '全幅', '幅', '寸法w'],
    'height': ['height', 'h', '全高', '高さ', '寸法h'],
    'weight': ['weight', 'wt', 'kg', '重量', '総重量', '質量'],
}


def detect_input_path(explicit_path: Optional[Path] = None) -> Path:
    search_roots = [Path.cwd(), *list(Path.cwd().parents)[:3]]
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        search_roots.append(drive_root)

    if explicit_path is not None:
        path = Path(explicit_path)
        if path.exists():
            return path

        fallback_name = path.name
        for root in search_roots:
            candidate = root / fallback_name
            if candidate.exists():
                return candidate

        if drive_root.exists():
            drive_matches = list(drive_root.rglob(fallback_name))
            if drive_matches:
                return drive_matches[0]

        raise FileNotFoundError(
            f'入力ファイルが見つかりません: {path} / cwd={Path.cwd()} にも {fallback_name} がありません。'
        )

    preferred_names = [
        '(抜粋)ケースリスト.csv',
        '(抜粋)ケースリスト.xlsx',
        'case_list.csv',
        'case_list.xlsx',
    ]

    for root in search_roots:
        for name in preferred_names:
            candidate = root / name
            if candidate.exists():
                return candidate
    if drive_root.exists():
        for name in preferred_names:
            drive_matches = list(drive_root.rglob(name))
            if drive_matches:
                return drive_matches[0]

    candidates: List[Path] = []
    for root in search_roots:
        for pattern in ('*.xlsx', '*.xls', '*.csv'):
            candidates.extend(sorted(root.glob(pattern)))

    if candidates:
        return list(dict.fromkeys(candidates))[0]

    raise FileNotFoundError(
        f'入力ファイルが見つかりません。cwd={Path.cwd()} / INPUT_PATH を確認してください。'
    )


def read_case_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        raw = pd.read_excel(path)
    elif suffix == '.csv':
        raw = pd.read_csv(path, encoding='utf-8-sig')
    else:
        raise ValueError(f'未対応のファイル形式です: {path.suffix}')

    if raw.empty:
        raise ValueError('入力ファイルが空です。')

    if raw.columns.str.contains('Unnamed').all() or any(str(c).startswith('Unnamed:') for c in raw.columns):
        first_row = raw.iloc[0].fillna('')
        if first_row.astype(str).str.len().gt(0).sum() >= 3:
            raw = raw.iloc[1:].copy()
            raw.columns = first_row.astype(str).tolist()

    renamed = {}
    normalized_columns = {col: normalize_column_name(col) for col in raw.columns}

    for canonical_name, aliases in COLUMN_CANDIDATES.items():
        alias_set = {normalize_column_name(a) for a in aliases}
        matched = [col for col, norm in normalized_columns.items() if norm in alias_set]
        if matched:
            renamed[matched[0]] = canonical_name

    df = raw.rename(columns=renamed).copy()

    required = ['case_id', 'length', 'width', 'height', 'weight']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError('必須列が不足しています。' + f' 不足列: {missing}. 利用可能列: {list(df.columns)}')

    df = df[required].copy()
    df = df.dropna(subset=required)
    df['case_id'] = df['case_id'].astype(str).str.strip()

    for col in ['length', 'width', 'height', 'weight']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if df[['length', 'width', 'height', 'weight']].isna().any().any():
        bad_rows = df[df[['length', 'width', 'height', 'weight']].isna().any(axis=1)]
        raise ValueError(f'数値変換に失敗した行があります。\n{bad_rows}')

    if (df[['length', 'width', 'height', 'weight']] <= 0).any().any():
        bad_rows = df[(df[['length', 'width', 'height', 'weight']] <= 0).any(axis=1)]
        raise ValueError(f'寸法または重量が0以下の行があります。\n{bad_rows}')

    if df['case_id'].duplicated().any():
        duplicated_ids = df.loc[df['case_id'].duplicated(), 'case_id'].tolist()
        raise ValueError(f'case_id が重複しています: {duplicated_ids}')

    return df.reset_index(drop=True)


INPUT_FILE = detect_input_path(INPUT_PATH)
INSTANCE_NAME = INSTANCE_NAME or INPUT_FILE.stem
case_df = read_case_table(INPUT_FILE)
display(case_df.head())
print(f'Loaded: {INPUT_FILE}')
print(f'Rows: {len(case_df)}')


FileNotFoundError: 入力ファイルが見つかりません: /content/drive/MyDrive/バンニングレイアウト/(抜粋)ケースリスト.csv / cwd=/content にも (抜粋)ケースリスト.csv がありません。

In [ ]:
# セル4: データモデル整形
@dataclass(frozen=True)
class CaseItem:
    case_id: str
    length: int
    width: int
    height: int
    weight: float
    volume: int


@dataclass(frozen=True)
class Placement:
    case_id: str
    container_id: int
    rotation: int
    x: int
    y: int
    z: int
    length: int
    width: int
    height: int
    weight: float

    @property
    def x2(self) -> int:
        return self.x + self.length

    @property
    def y2(self) -> int:
        return self.y + self.width

    @property
    def z2(self) -> int:
        return self.z + self.height

    @property
    def center_x(self) -> float:
        return self.x + self.length / 2.0

    @property
    def center_y(self) -> float:
        return self.y + self.width / 2.0

    @property
    def center_z(self) -> float:
        return self.z + self.height / 2.0


@dataclass
class ContainerState:
    container_id: int
    placements: List[Placement] = field(default_factory=list)

    @property
    def total_weight(self) -> float:
        return float(sum(p.weight for p in self.placements))

    @property
    def total_volume(self) -> int:
        return int(sum(p.length * p.width * p.height for p in self.placements))


def build_cases(df: pd.DataFrame) -> List[CaseItem]:
    cases = []
    for row in df.itertuples(index=False):
        cases.append(
            CaseItem(
                case_id=row.case_id,
                length=int(row.length),
                width=int(row.width),
                height=int(row.height),
                weight=float(row.weight),
                volume=int(row.length * row.width * row.height),
            )
        )

    return sorted(cases, key=lambda c: (-c.weight, -c.volume, -max(c.length, c.width, c.height), c.case_id))


cases = build_cases(case_df)
display(pd.DataFrame([asdict(c) for c in cases]).head())
print(f'Total cases: {len(cases)}')


In [ ]:
# セル5: 幾何判定関数
def get_rotated_dims(case: CaseItem, rotation: int) -> Tuple[int, int, int]:
    if rotation == 0:
        return case.length, case.width, case.height
    if rotation == 90:
        return case.width, case.length, case.height
    raise ValueError(f'Unsupported rotation: {rotation}')


def fits_in_container(x: int, y: int, z: int, length: int, width: int, height: int) -> bool:
    return (
        x >= 0 and y >= 0 and z >= 0 and
        x + length <= CONTAINER_LENGTH_MM and
        y + width <= CONTAINER_WIDTH_MM and
        z + height <= CONTAINER_HEIGHT_MM
    )


def boxes_overlap(a: Placement, b: Placement) -> bool:
    return not (
        a.x2 <= b.x or b.x2 <= a.x or
        a.y2 <= b.y or b.y2 <= a.y or
        a.z2 <= b.z or b.z2 <= a.z
    )


def is_fully_supported(candidate: Placement, existing: Sequence[Placement]) -> bool:
    if candidate.z == 0:
        return True

    for p in existing:
        if (
            p.z2 == candidate.z and
            p.x <= candidate.x and candidate.x2 <= p.x2 and
            p.y <= candidate.y and candidate.y2 <= p.y2
        ):
            return True
    return False


def compute_center_of_gravity(placements: Sequence[Placement]) -> Tuple[float, float, float]:
    if not placements:
        return CONTAINER_CENTER_X_MM, CONTAINER_CENTER_Y_MM, 0.0

    total_w = sum(p.weight for p in placements)
    cx = sum(p.center_x * p.weight for p in placements) / total_w
    cy = sum(p.center_y * p.weight for p in placements) / total_w
    cz = sum(p.center_z * p.weight for p in placements) / total_w
    return cx, cy, cz


def center_of_gravity_radius(placements: Sequence[Placement]) -> float:
    cx, cy, _ = compute_center_of_gravity(placements)
    return math.hypot(cx - CONTAINER_CENTER_X_MM, cy - CONTAINER_CENTER_Y_MM)


def satisfies_center_of_gravity(placements: Sequence[Placement]) -> bool:
    return center_of_gravity_radius(placements) <= COG_RADIUS_LIMIT_MM + 1e-9


def build_placement(case: CaseItem, container_id: int, rotation: int, x: int, y: int, z: int) -> Placement:
    length, width, height = get_rotated_dims(case, rotation)
    return Placement(case.case_id, container_id, rotation, int(x), int(y), int(z), int(length), int(width), int(height), float(case.weight))


def can_place(container: ContainerState, candidate: Placement) -> bool:
    if not fits_in_container(candidate.x, candidate.y, candidate.z, candidate.length, candidate.width, candidate.height):
        return False

    if any(boxes_overlap(candidate, placed) for placed in container.placements):
        return False

    if not is_fully_supported(candidate, container.placements):
        return False

    if container.total_weight + candidate.weight > MAX_CONTAINER_WEIGHT_KG + 1e-9:
        return False

    if not satisfies_center_of_gravity([*container.placements, candidate]):
        return False

    return True


In [ ]:
# セル6: 配置アルゴリズム本体
def generate_candidate_points(container: ContainerState) -> List[Tuple[int, int, int]]:
    points = {(0, 0, 0)}
    for p in container.placements:
        points.add((p.x2, p.y, p.z))
        points.add((p.x, p.y2, p.z))
        points.add((p.x, p.y, p.z2))
        points.add((p.x2, p.y2, p.z))
        points.add((p.x2, p.y, p.z2))
        points.add((p.x, p.y2, p.z2))
    return sorted([pt for pt in points if pt[0] <= CONTAINER_LENGTH_MM and pt[1] <= CONTAINER_WIDTH_MM and pt[2] <= CONTAINER_HEIGHT_MM], key=lambda t: (t[2], t[0], t[1]))


def centered_floor_point(length: int, width: int) -> Tuple[int, int, int]:
    x = max(0, min(CONTAINER_LENGTH_MM - length, int(round(CONTAINER_CENTER_X_MM - length / 2.0))))
    y = max(0, min(CONTAINER_WIDTH_MM - width, int(round(CONTAINER_CENTER_Y_MM - width / 2.0))))
    return x, y, 0


def bounding_box_volume(placements: Sequence[Placement]) -> int:
    if not placements:
        return 0
    return max(p.x2 for p in placements) * max(p.y2 for p in placements) * max(p.z2 for p in placements)


def evaluate_candidate(container: ContainerState, candidate: Placement) -> Tuple[float, float, float, int, int, int]:
    new_placements = [*container.placements, candidate]
    cog_radius = center_of_gravity_radius(new_placements)
    used_volume = sum(p.length * p.width * p.height for p in new_placements)
    dead_space = bounding_box_volume(new_placements) - used_volume
    return (cog_radius, dead_space, candidate.z, candidate.x, candidate.y, candidate.rotation)


def find_best_placement(container: ContainerState, case: CaseItem) -> Optional[Placement]:
    best_candidate: Optional[Placement] = None
    best_score: Optional[Tuple[float, float, float, int, int, int]] = None
    base_points = generate_candidate_points(container)

    for rotation in ROTATIONS:
        length, width, height = get_rotated_dims(case, rotation)
        point_candidates = sorted(set([*base_points, centered_floor_point(length, width)]), key=lambda t: (t[2], t[0], t[1]))
        for x, y, z in point_candidates:
            candidate = build_placement(case, container.container_id, rotation, x, y, z)
            if not can_place(container, candidate):
                continue

            score = evaluate_candidate(container, candidate)
            if best_score is None or score < best_score:
                best_candidate = candidate
                best_score = score

    return best_candidate


def pack_cases(case_list: Sequence[CaseItem]) -> List[ContainerState]:
    containers: List[ContainerState] = []
    for case in case_list:
        placed = False
        for container in containers:
            candidate = find_best_placement(container, case)
            if candidate is not None:
                container.placements.append(candidate)
                placed = True
                break

        if placed:
            continue

        new_container = ContainerState(container_id=len(containers) + 1)
        candidate = find_best_placement(new_container, case)
        if candidate is None:
            raise RuntimeError(f'ケース {case.case_id} を単独でも配置できません。寸法または重心条件を確認してください。')
        new_container.placements.append(candidate)
        containers.append(new_container)

    return containers


containers = pack_cases(cases)
all_placements = [p for container in containers for p in container.placements]
print(f'Containers used: {len(containers)}')
print(f'Placements created: {len(all_placements)}')


In [ ]:
# セル7: 評価関数
def evaluate_solution(cases: Sequence[CaseItem], containers: Sequence[ContainerState]) -> Dict[str, object]:
    case_map = {c.case_id: c for c in cases}
    placements = [p for container in containers for p in container.placements]
    violations: List[str] = []

    placed_ids = [p.case_id for p in placements]
    if len(placed_ids) != len(case_map):
        violations.append(f'配置件数不一致: expected={len(case_map)}, actual={len(placed_ids)}')

    duplicates = pd.Series(placed_ids).value_counts()
    duplicate_ids = duplicates[duplicates > 1].index.tolist()
    if duplicate_ids:
        violations.append(f'重複配置あり: {duplicate_ids}')

    missing_ids = sorted(set(case_map) - set(placed_ids))
    if missing_ids:
        violations.append(f'未配置ケースあり: {missing_ids}')

    container_summaries = []
    max_cog_radius = 0.0

    for container in containers:
        for p in container.placements:
            if p.rotation not in ROTATIONS:
                violations.append(f'不正rotation: case_id={p.case_id}, rotation={p.rotation}')
            if not fits_in_container(p.x, p.y, p.z, p.length, p.width, p.height):
                violations.append(f'コンテナ外はみ出し: case_id={p.case_id}')
            if not is_fully_supported(p, [q for q in container.placements if q.case_id != p.case_id]):
                violations.append(f'支持条件違反: case_id={p.case_id}')

        for i, a in enumerate(container.placements):
            for b in container.placements[i + 1:]:
                if boxes_overlap(a, b):
                    violations.append(f'重なりあり: {a.case_id} vs {b.case_id} in container {container.container_id}')

        weight = container.total_weight
        if weight > MAX_CONTAINER_WEIGHT_KG + 1e-9:
            violations.append(f'重量超過: container {container.container_id}, weight={weight}')

        cog = compute_center_of_gravity(container.placements)
        cog_radius = center_of_gravity_radius(container.placements)
        if cog_radius > COG_RADIUS_LIMIT_MM + 1e-9:
            violations.append(f'重心制約違反: container {container.container_id}, radius={cog_radius:.2f}mm')

        max_cog_radius = max(max_cog_radius, cog_radius)
        container_summaries.append({
            'container_id': container.container_id,
            'case_count': len(container.placements),
            'weight_kg': round(weight, 3),
            'center_of_gravity_mm': tuple(round(v, 3) for v in cog),
            'cog_radius_mm': round(cog_radius, 3),
            'fill_rate': round(container.total_volume / CONTAINER_VOLUME_MM3, 6),
        })

    return {
        'valid': len(violations) == 0,
        'violations': violations,
        'container_count': len(containers),
        'container_summaries': container_summaries,
        'max_cog_radius_mm': round(max_cog_radius, 3),
        'overall_fill_rate': round(sum(c.total_volume for c in containers) / max(len(containers), 1) / CONTAINER_VOLUME_MM3, 6),
    }


evaluation = evaluate_solution(cases, containers)
print(json.dumps(evaluation, ensure_ascii=False, indent=2))


In [ ]:
# セル8: JSON出力
solution = {
    'metadata': {
        'instance_name': INSTANCE_NAME,
        'algorithm': ALGORITHM_NAME,
        'created_at': datetime.now(timezone.utc).isoformat(),
    },
    'placements': sorted(
        [
            {
                'case_id': p.case_id,
                'container_id': p.container_id,
                'rotation': p.rotation,
                'x': p.x,
                'y': p.y,
                'z': p.z,
            }
            for p in all_placements
        ],
        key=lambda item: item['case_id'],
    ),
}

with OUTPUT_JSON_PATH.open('w', encoding='utf-8') as f:
    json.dump(solution, f, ensure_ascii=False, indent=2)

print(f'JSON saved to: {OUTPUT_JSON_PATH.resolve()}')
display(pd.DataFrame(solution['placements']).head())


In [ ]:
# セル9: 結果表示
if 'evaluation' not in globals():
    raise RuntimeError('evaluation が未作成です。セル7までを上から順に実行してください。')

summary_df = pd.DataFrame(evaluation['container_summaries'])

print(f"使用コンテナ本数: {evaluation['container_count']}")
print(f"最大重心距離(mm): {evaluation['max_cog_radius_mm']}")
print(f"制約違反有無: {'なし' if evaluation['valid'] else 'あり'}")
print(f"JSON保存先: {OUTPUT_JSON_PATH.resolve()}")

if evaluation['violations']:
    print('\n制約違反詳細:')
    for violation in evaluation['violations']:
        print('-', violation)

if not summary_df.empty:
    display(summary_df)
else:
    print('コンテナサマリは空です。')


## 改善案

1. 局所探索でケース順序や回転を微調整する。
2. いったん配置済みケースを抜いて詰め直す destroy-and-repair を入れる。
3. コンテナ間 swap で重心距離や本数を改善する。
4. 候補点生成を強化し、端点の組み合わせも探索する。
5. ビームサーチや焼きなましに拡張して初期順序依存を弱める。